# 07 Load Adapters from MLflow to vLLM

This notebook documents the dynamic adapter loading.

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Download adapters from MLFlow

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(settings_cfg.mlflow_tracking_uri)
mlflow.set_experiment(settings_cfg.mlflow_experiment_name)
client = MlflowClient()
experiment = client.get_experiment_by_name(settings_cfg.mlflow_experiment_name)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else "not created yet")

In [ ]:
from pathlib import Path
import shutil
import mlflow
from mlflow.artifacts import download_artifacts

# Fetch MLflow runs
runs = mlflow.search_runs()

adapter_root = Path("/home/vinchar/llmops-demo/adapters")
adapter_root.mkdir(parents=True, exist_ok=True)

downloaded_adapters = {}

for adapter in settings_cfg.adapters:

    adapter_runs = runs[
        (runs["tags.adapter_name"] == adapter)
        & (runs["status"] == "FINISHED")
    ].sort_values("start_time", ascending=False)

    if adapter_runs.empty:
        print(f"No successful run found for {adapter}")
        continue

    run_id = adapter_runs.iloc[0]["run_id"]

    print(f"Downloading '{adapter}' from run {run_id}")

    # Final desired location
    final_dir = adapter_root / adapter

    # Remove old adapter dir if it exists
    if final_dir.exists():
        shutil.rmtree(final_dir)

    # Download artifacts
    downloaded_path = download_artifacts(
        artifact_uri=f"runs:/{run_id}/adapter",
        dst_path=str(adapter_root),
    )

    downloaded_path = Path(downloaded_path)

    # Sometimes MLflow downloads as adapter/adapter/*
    nested = downloaded_path / "adapter"

    if nested.exists():
        final_dir.mkdir(parents=True, exist_ok=True)

        for item in nested.iterdir():
            shutil.move(str(item), str(final_dir))

        nested.rmdir()

        # remove empty parent dir
        if downloaded_path.exists() and downloaded_path != final_dir:
            shutil.rmtree(downloaded_path)

    else:
        shutil.move(str(downloaded_path), str(final_dir))

    downloaded_adapters[adapter] = final_dir

    print(f"Downloaded {adapter} to {final_dir}")

print("\nFinal adapter locations:")
for name, path in downloaded_adapters.items():
    print(f"{name}: {path}")

## Copy adapters to vLLM Model

In [ ]:
!/home/vinchar/kube cp \
  /home/vinchar/llmops-demo/adapters \
  project-public/qwen257b-predictor-00001-deployment-59df6df457-gwj2k:/tmp/adapters \
  -c kserve-container

In [ ]:
!/home/vinchar/kube exec -it \
  -n project-public \
  qwen257b-predictor-00001-deployment-59df6df457-gwj2k \
  -c kserve-container -- \
  bash -c "ls -l /tmp/adapters/"

## Load adapters

Expected output: one success line per adapter.

In [ ]:
from scripts.load_adapters import load_adapter

for adapter in settings_cfg.adapters:

    try:
        print(f"Loading adapter: {adapter}")

        load_adapter(
            settings_cfg.vllm_base_url,
            settings_cfg.vllm_api_key,
            adapter,
            Path(f"/tmp/adapters/{adapter}"),
        )

    except RuntimeError as e:

        if "already been loaded" in str(e):
            print(f"{adapter} already loaded, skipping.")
        else:
            raise

print("Done.")

## Verify vLLM model registration

Expected output: `finance`, `legal`, and `healthcare` appear in `/v1/models`.

In [ ]:
import requests

response = requests.get(
    f"{settings_cfg.vllm_base_url}/v1/models",
    headers={"Authorization": f"Bearer {settings_cfg.vllm_api_key}"},
    timeout=10,
)
response.raise_for_status()
print(response.json())